# MetaboNet — Live Leaderboard (Final Production Version)

In [1]:
import subprocess, sys
def pip_q(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
pip_q('lightgbm>=4.0.0')
pip_q('polars>=0.20.0')
pip_q('pyarrow>=14.0.0')
print('Dependencies ready.')

Dependencies ready.


In [2]:
import gc, math, os, shutil, time, warnings
import numpy as np
import pandas as pd
import polars as pl
from pathlib import Path
import lightgbm as lgb
warnings.filterwarnings('ignore')

# ── Paths (exact Kaggle mount points from dataset screenshot) ──────────────
TRAIN_PARQUET      = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet')
TEST_PARQUET       = Path('/kaggle/input/datasets/prosenjitmondol/metabonet/test.parquet')
LIVE_TEMPLATE_PATH = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/template.parquet')
LIVE_TARGETS_PATH  = Path('/kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/targets.parquet')
OUTPUT_DIR = Path('/kaggle/working')
SHARD_DIR  = OUTPUT_DIR / 'shards'
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Fallback path detection (in case mount name differs) ──────────────────
if not TRAIN_PARQUET.exists():
    candidates = [
        '/kaggle/input/datasets/prosenjitmondol/metabonet',
        '/kaggle/input/t1d-challenge',
        '/kaggle/input/metabonet-glucose',
    ]
    for base in candidates:
        if (Path(base) / 'train.parquet').exists():
            TRAIN_PARQUET = Path(base) / 'train.parquet'
            TEST_PARQUET  = Path(base) / 'test.parquet'
            break
    if not TRAIN_PARQUET.exists():
        for f in Path('/kaggle/input').rglob('train.parquet'):
            TRAIN_PARQUET = f
            TEST_PARQUET  = f.parent / 'test.parquet'
            break

if not LIVE_TEMPLATE_PATH.exists():
    import pyarrow.parquet as pq
    for f in Path('/kaggle/input').rglob('template.parquet'):
        try:
            if pq.read_metadata(f).num_rows > 1_000_000:
                LIVE_TEMPLATE_PATH = f
                break
        except Exception:
            pass

if not LIVE_TARGETS_PATH.exists():
    for f in Path('/kaggle/input').rglob('targets.parquet'):
        LIVE_TARGETS_PATH = f
        break

print(f'train     : {TRAIN_PARQUET} → {TRAIN_PARQUET.exists()}')
print(f'test      : {TEST_PARQUET}  → {TEST_PARQUET.exists()}')
print(f'template  : {LIVE_TEMPLATE_PATH} → {LIVE_TEMPLATE_PATH.exists()}')
print(f'targets   : {LIVE_TARGETS_PATH} → {LIVE_TARGETS_PATH.exists()}')

assert TRAIN_PARQUET.exists(),      f'train.parquet not found: {TRAIN_PARQUET}'
assert TEST_PARQUET.exists(),       f'test.parquet not found: {TEST_PARQUET}'
assert LIVE_TEMPLATE_PATH.exists(), f'Live template not found: {LIVE_TEMPLATE_PATH}'

# Load template with pandas (MUST use pandas — matches leaderboard validation)
live_template = pd.read_parquet(LIVE_TEMPLATE_PATH)
live_template['date'] = pd.to_datetime(live_template['date'])
EXPECTED_ROWS = len(live_template)
assert EXPECTED_ROWS > 1_000_000, f'Wrong template: {EXPECTED_ROWS} rows (expected 2,648,987)'
assert list(live_template.columns) == ['id','source_file','date','pred_30','pred_60','pred_90','pred_120'], \
    f'Unexpected columns: {list(live_template.columns)}'
print(f'\nLive template: {EXPECTED_ROWS:,} rows | {live_template["id"].nunique()} patients')
print(live_template.head(3).to_string())

# ── Hyperparameters ───────────────────────────────────────────────────────
HORIZONS         = [30, 60, 90, 120]
CGM_INTERVAL     = 5        # minutes between CGM readings
GLUCOSE_MIN      = 39.0     # physiological floor (mg/dL)
GLUCOSE_MAX      = 500.0    # physiological ceiling (mg/dL)
MAX_ROC_PER_MIN  = 4.0      # max allowed rate of change (mg/dL/min)
SEED             = 42
PATIENT_BATCH    = 25       # patients processed per shard
TRAIN_STRIDE     = 12       # 1-in-12 rows: ~9M rows, safe for 16 GB Kaggle RAM
IOB_DECAY_LAMBDA = 0.0025
IOB_WINDOW_STEPS = 48       # 4h insulin window
COB_WINDOW_STEPS = 36       # 3h carb window
COB_PEAK_STEP    = 9        # carb absorption peak at 45 min

# LightGBM: MAE objective → directly minimises MARD
LGBM_PARAMS = {
    'objective'       : 'regression_l1',  # MAE loss ≈ MARD
    'metric'          : 'mae',
    'verbosity'       : -1,
    'num_leaves'      : 127,
    'max_depth'       : 9,
    'max_bin'         : 255,
    'learning_rate'   : 0.04,
    'feature_fraction': 0.75,
    'bagging_fraction': 0.80,
    'bagging_freq'    : 1,
    'min_child_samples': 30,
    'lambda_l1'       : 0.05,
    'lambda_l2'       : 0.10,
    'n_jobs'          : 4,
    'seed'            : SEED,
    'force_col_wise'  : True,
}

def rmse(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p) | np.isnan(t))
    return round(float(np.sqrt(np.mean((p[m]-t[m])**2))), 2)

def mard(p, t):
    p, t = np.asarray(p, float), np.asarray(t, float)
    m = ~(np.isnan(p) | np.isnan(t)) & (t > 0)
    return round(float(np.mean(np.abs(p[m]-t[m])/t[m]) * 100), 2)

print('\nConfiguration loaded. Ready to run.')

train     : /kaggle/input/datasets/prosenjitmondol/metabonet/train.parquet → True
test      : /kaggle/input/datasets/prosenjitmondol/metabonet/test.parquet  → True
template  : /kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/template.parquet → True
targets   : /kaggle/input/datasets/prosenjitmondol/metabonet-toolkit-annual/live-leaderboard/targets.parquet → True

Live template: 2,648,987 rows | 268 patients
   id source_file                date  pred_30  pred_60  pred_90  pred_120
0  16       AZT1D 2024-02-02 21:55:00      NaN      NaN      NaN       NaN
1  16       AZT1D 2024-02-02 22:10:00      NaN      NaN      NaN       NaN
2  16       AZT1D 2024-02-02 22:25:00      NaN      NaN      NaN       NaN

Configuration loaded. Ready to run.


In [3]:
# ===========================================================================
# CELL 3: Feature Engineering Pipeline
# ===========================================================================
import math as _math

def _build_iob(steps, lam):
    return np.exp(-lam * np.arange(steps) * CGM_INTERVAL).astype(np.float32)

def _build_cob(steps, peak):
    t = np.arange(steps, dtype=np.float32)
    return np.where(t <= peak, t / max(peak, 1),
                    np.exp(-0.015 * (t - peak) * CGM_INTERVAL)).astype(np.float32)

_IOB = _build_iob(IOB_WINDOW_STEPS, IOB_DECAY_LAMBDA)
_COB = _build_cob(COB_WINDOW_STEPS, COB_PEAK_STEP)

# Rolling window sizes — module-level so accessible in print statements
WIN_SIZES  = [3, 6, 12, 24, 48]
WIN_LABELS = ['15m', '30m', '1h', '2h', '4h']

def _conv1d(kernel, data):
    """Causal convolution (no future leakage)."""
    k = len(kernel)
    padded = np.concatenate([np.zeros(k - 1, dtype=np.float32), data])
    return np.convolve(padded, kernel, 'valid')[:len(data)]

def _add_iob_cob(df: pd.DataFrame) -> pd.DataFrame:
    """Add Insulin-on-Board and Carbs-on-Board columns per patient."""
    for src_col, out_col, kernel in [
        ('insulin', 'iob_total', _IOB),
        ('bolus',   'iob_bolus', _IOB),
        ('basal',   'iob_basal', _IOB),
    ]:
        if src_col in df.columns:
            segs = [
                _conv1d(kernel, g[src_col].fillna(0).to_numpy(np.float32))
                for _, g in df.groupby('id', sort=False)
            ]
            df[out_col] = np.concatenate(segs).astype(np.float32)
        else:
            df[out_col] = np.float32(0.0)

    if 'carbs' in df.columns:
        segs = [
            _conv1d(_COB, g['carbs'].fillna(0).to_numpy(np.float32))
            for _, g in df.groupby('id', sort=False)
        ]
        df['cob_total'] = np.concatenate(segs).astype(np.float32)
    else:
        df['cob_total'] = np.float32(0.0)
    return df


def _featurise_lazy(lazy: pl.LazyFrame, is_train: bool, stride: int) -> pd.DataFrame:
    """Convert raw CGM lazy frame → feature DataFrame."""
    schema = set(lazy.collect_schema().names())

    want = ['id', 'date', 'source_file', 'CGM', 'basal', 'bolus',
            'insulin', 'carbs', 'age', 'weight', 'height',
            'gender', 'age_of_diagnosis']
    lazy = lazy.select([c for c in want if c in schema])
    lazy = lazy.with_columns(pl.col('date').cast(pl.Datetime('us')))

    # --- CGM cleaning ---------------------------------------------------------
    # Mask tokens: -1 (CGM_MASK_TOKEN) and -2 (CGM_IMPUTED_TOKEN) → null
    lazy = lazy.with_columns(
        pl.when(pl.col('CGM').is_null() | (pl.col('CGM') <= 0))
          .then(pl.lit(1, dtype=pl.Int8))
          .otherwise(pl.lit(0, dtype=pl.Int8))
          .alias('cgm_is_missing')
    )
    lazy = lazy.with_columns(
        pl.when(pl.col('CGM') > 0)
          .then(pl.col('CGM'))
          .otherwise(None)
          .cast(pl.Float32)
          .alias('CGM_clean')
    )
    # Forward/back fill then clip to physiological range
    lazy = lazy.with_columns(
        pl.col('CGM_clean')
          .forward_fill()
          .backward_fill()
          .fill_null(120.0)
          .over('id')
          .clip(GLUCOSE_MIN, GLUCOSE_MAX)
          .alias('CGM_clean')
    )

    PI   = _math.pi
    exprs = []

    # --- Targets (train only) -------------------------------------------------
    if is_train:
        for h in HORIZONS:
            exprs.append(
                pl.col('CGM_clean')
                  .shift(-(h // CGM_INTERVAL))
                  .over('id')
                  .alias(f'target_{h}')
            )

    # --- CGM lag features (5 min to 8 hours) ----------------------------------
    for s in [1, 2, 3, 4, 5, 6, 9, 12, 18, 24, 36, 48, 72, 96]:
        exprs.append(
            pl.col('CGM_clean').shift(s).over('id').cast(pl.Float32).alias(f'cgm_lag_{s}')
        )

    # --- Rate of change (velocity) --------------------------------------------
    for s in [1, 2, 3, 6, 12]:
        exprs.append(
            ((pl.col('CGM_clean') - pl.col('CGM_clean').shift(s)).over('id') / s)
            .cast(pl.Float32).alias(f'cgm_roc_{s}')
        )

    # --- Acceleration (2nd derivative) ----------------------------------------
    exprs.append(
        (
            pl.col('CGM_clean')
            - 2 * pl.col('CGM_clean').shift(1)
            + pl.col('CGM_clean').shift(2)
        ).over('id').cast(pl.Float32).alias('cgm_accel')
    )

    # --- Rolling statistics (15m, 30m, 1h, 2h, 4h) ---------------------------
    # WIN_SIZES / WIN_LABELS defined at module level (Cell 3 top)
    for w, lbl in zip(WIN_SIZES, WIN_LABELS):
        base = pl.col('CGM_clean').over('id')
        exprs += [
            base.rolling_mean(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_mean'),
            base.rolling_std(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_std'),
            base.rolling_min(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_min'),
            base.rolling_max(window_size=w).cast(pl.Float32).alias(f'cgm_{lbl}_max'),
        ]

    # --- Consecutive missing count (1-hour window) ----------------------------
    exprs.append(
        pl.col('cgm_is_missing')
          .rolling_sum(window_size=12)
          .over('id')
          .cast(pl.Int8)
          .alias('cgm_missing_1h')
    )

    # --- Insulin / carb summaries ---------------------------------------------
    for col, alias, win in [
        ('insulin', 'insulin_30m', 6),
        ('insulin', 'insulin_1h',  12),
        ('bolus',   'bolus_30m',   6),
        ('carbs',   'carbs_1h',    12),
        ('carbs',   'carbs_2h',    24),
    ]:
        if col in schema:
            exprs.append(
                pl.col(col).fill_null(0)
                  .rolling_sum(window_size=win)
                  .over('id')
                  .cast(pl.Float32)
                  .alias(alias)
            )

    for col, alias, win in [
        ('carbs', 'had_carbs_30m', 6),
        ('carbs', 'had_carbs_1h',  12),
    ]:
        if col in schema:
            exprs.append(
                (pl.col(col).fill_null(0).rolling_sum(window_size=win).over('id') > 0)
                .cast(pl.Int8).alias(alias)
            )

    # --- Time-of-day features -------------------------------------------------
    mins_col = (pl.col('date').dt.hour() * 60 + pl.col('date').dt.minute()).cast(pl.Float32)
    exprs += [
        pl.col('date').dt.hour().cast(pl.Int8).alias('hour_of_day'),
        (mins_col * (2 * PI / 1440)).sin().cast(pl.Float32).alias('time_sin'),
        (mins_col * (2 * PI / 1440)).cos().cast(pl.Float32).alias('time_cos'),
        (mins_col * (4 * PI / 1440)).sin().cast(pl.Float32).alias('time_sin2'),
        (mins_col * (4 * PI / 1440)).cos().cast(pl.Float32).alias('time_cos2'),
        pl.col('date').dt.weekday().cast(pl.Int8).alias('day_of_week'),
        (pl.col('date').dt.weekday() >= 5).cast(pl.Int8).alias('is_weekend'),
        # Physiological periods
        ((pl.col('date').dt.hour() >= 4) & (pl.col('date').dt.hour() < 9)).cast(pl.Int8).alias('is_dawn'),
        ((pl.col('date').dt.hour() >= 22) | (pl.col('date').dt.hour() < 6)).cast(pl.Int8).alias('is_night'),
        ((pl.col('date').dt.hour() >= 11) & (pl.col('date').dt.hour() < 14)).cast(pl.Int8).alias('is_lunch'),
        ((pl.col('date').dt.hour() >= 17) & (pl.col('date').dt.hour() < 20)).cast(pl.Int8).alias('is_dinner'),
    ]

    # --- Patient demographics -------------------------------------------------
    if 'source_file' in schema:
        exprs.append(
            pl.col('source_file').cast(pl.Categorical).to_physical()
              .cast(pl.Int16).alias('source_code')
        )
    if 'weight' in schema and 'height' in schema:
        exprs.append(
            (pl.col('weight') / ((pl.col('height') / 100) ** 2))
            .cast(pl.Float32).alias('bmi')
        )
    if 'age' in schema and 'age_of_diagnosis' in schema:
        exprs.append(
            (pl.col('age') - pl.col('age_of_diagnosis'))
            .clip(0, 80).cast(pl.Float32).alias('diabetes_duration')
        )
    if 'gender' in schema:
        exprs.append(
            pl.when(pl.col('gender').cast(pl.String).str.to_lowercase() == 'male').then(pl.lit(1, pl.Int8))
            .when(pl.col('gender').cast(pl.String).str.to_lowercase() == 'female').then(pl.lit(0, pl.Int8))
            .otherwise(pl.lit(-1, pl.Int8))
            .alias('gender_code')
        )

    # --- Glucose zone flags ---------------------------------------------------
    exprs += [
        (pl.col('CGM_clean') < 70).cast(pl.Int8).alias('is_hypo'),
        ((pl.col('CGM_clean') >= 70) & (pl.col('CGM_clean') <= 180)).cast(pl.Int8).alias('in_range'),
        (pl.col('CGM_clean') > 180).cast(pl.Int8).alias('is_hyper'),
    ]

    lazy = lazy.with_columns(exprs)

    # --- Derived features (second pass, need first-pass columns) -------------
    exprs2 = []
    for lbl in WIN_LABELS:
        exprs2.append(
            (pl.col(f'cgm_{lbl}_max') - pl.col(f'cgm_{lbl}_min'))
            .cast(pl.Float32).alias(f'cgm_{lbl}_range')
        )
    # CGM trend direction code (0=falling fast … 4=rising fast)
    exprs2.append(
        pl.when(pl.col('cgm_roc_3') > 1.0).then(pl.lit(4, pl.Int8))
        .when(pl.col('cgm_roc_3') > 0.3).then(pl.lit(3, pl.Int8))
        .when(pl.col('cgm_roc_3') >= -0.3).then(pl.lit(2, pl.Int8))
        .when(pl.col('cgm_roc_3') >= -1.0).then(pl.lit(1, pl.Int8))
        .otherwise(pl.lit(0, pl.Int8))
        .alias('cgm_trend_code')
    )
    lazy = lazy.with_columns(exprs2)

    # --- Training-specific: filter + stride -----------------------------------
    if is_train:
        # Only keep rows where CGM_clean is valid (i.e., not imputed)
        # Note: CGM_clean was forward/back filled, so filter on original cgm_is_missing
        lazy = lazy.filter(pl.col('cgm_is_missing') == 0)
        # Apply stride using row_number within each patient
        if stride > 1:
            lazy = lazy.with_columns(
                pl.col('id').cum_count().over('id').cast(pl.Int32).alias('_rn')
            )
            lazy = lazy.filter(pl.col('_rn') % stride == 0).drop('_rn')

    df = lazy.collect().to_pandas()
    df['date'] = pd.to_datetime(df['date'])
    return df


print('Feature pipeline defined.')
print(f'  LAG features  : {len([1,2,3,4,5,6,9,12,18,24,36,48,72,96])} columns (5m–8h)')
print(f'  ROC features  : {len([1,2,3,6,12])} columns')
print(f'  Rolling stats : {len(WIN_SIZES)*4} columns (mean/std/min/max × 5 windows)')
print(f'  Rolling range : {len(WIN_SIZES)} columns')

Feature pipeline defined.
  LAG features  : 14 columns (5m–8h)
  ROC features  : 5 columns
  Rolling stats : 20 columns (mean/std/min/max × 5 windows)
  Rolling range : 5 columns


In [4]:
# ===========================================================================
# CELL 4: Extract Training Features + Compute Patient Statistics
# ===========================================================================
if SHARD_DIR.exists():
    shutil.rmtree(SHARD_DIR)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

print(f'Streaming train.parquet → shards (stride={TRAIN_STRIDE})...')
t0 = time.time()

all_pts = pl.scan_parquet(TRAIN_PARQUET).select('id').unique().collect()['id'].to_list()
chunks  = [all_pts[i:i + PATIENT_BATCH] for i in range(0, len(all_pts), PATIENT_BATCH)]
print(f'  Total patients: {len(all_pts):,} | Batches: {len(chunks)}')

shard_paths = []
total_rows  = 0
for idx, batch in enumerate(chunks):
    lazy  = pl.scan_parquet(TRAIN_PARQUET).filter(pl.col('id').is_in(batch))
    pdf   = _featurise_lazy(lazy, is_train=True, stride=TRAIN_STRIDE)
    pdf   = _add_iob_cob(pdf)
    shard = SHARD_DIR / f'shard_{idx:04d}.parquet'
    pdf.to_parquet(shard, index=False)
    shard_paths.append(shard)
    total_rows += len(pdf)
    del pdf, lazy
    gc.collect()
    if (idx + 1) % max(1, len(chunks) // 5) == 0 or idx + 1 == len(chunks):
        print(f'  Batch {idx+1:3d}/{len(chunks)} | {total_rows:,} rows | [{time.time()-t0:.0f}s]')

print(f'\nDone: {len(shard_paths)} shards | {total_rows:,} training rows')

# Determine feature columns from first shard
_sample_cols = list(pd.read_parquet(shard_paths[0]).columns)
EXCLUDE = {
    'id', 'date', 'source_file', 'CGM', 'CGM_clean',
    'gender', 'meal_label', 'workout_label', 'cgm_device', 'misc_notes',
    'insulin_delivery_algorithm', 'insulin_delivery_device', 'insulin_delivery_modality',
    'insulin_type_basal', 'insulin_type_bolus', 'ethnicity', 'treatment_group',
    'randomization_date', 'extension_date', 'is_test', 'is_pregnant',
    'subject_split_across_traintest',
    'target_30', 'target_60', 'target_90', 'target_120',
}
FEATURE_COLS = sorted([c for c in _sample_cols if c not in EXCLUDE])
print(f'\nFeature columns: {len(FEATURE_COLS)}')
print(f'  First 10: {FEATURE_COLS[:10]}')

# Compute per-patient CGM statistics (used as features at inference)
print('\nComputing per-patient statistics...')
pat_stats_list = []
for sp in shard_paths:
    tmp = pd.read_parquet(sp, columns=['id', 'CGM_clean'])
    ps  = tmp.groupby('id')['CGM_clean'].agg(['mean', 'std', 'min', 'max']).reset_index()
    ps.columns = ['id', 'pat_cgm_mean', 'pat_cgm_std', 'pat_cgm_min', 'pat_cgm_max']
    pat_stats_list.append(ps)
    del tmp

pat_stats = pd.concat(pat_stats_list).groupby('id').mean().reset_index()
pat_stats['pat_cgm_std'] = pat_stats['pat_cgm_std'].fillna(30.0)
PAT_FEAT_COLS = ['pat_cgm_mean', 'pat_cgm_std', 'pat_cgm_min', 'pat_cgm_max']
ALL_FEATURE_COLS = FEATURE_COLS + PAT_FEAT_COLS
print(f'  {len(pat_stats)} patients with statistics')
print(f'  Total features with patient stats: {len(ALL_FEATURE_COLS)}')
del pat_stats_list
gc.collect()

Streaming train.parquet → shards (stride=12)...
  Total patients: 1,183 | Batches: 48
  Batch   9/48 | 1,799,293 rows | [46s]
  Batch  18/48 | 3,539,702 rows | [82s]
  Batch  27/48 | 5,282,532 rows | [117s]
  Batch  36/48 | 7,124,455 rows | [153s]
  Batch  45/48 | 8,809,378 rows | [185s]
  Batch  48/48 | 9,275,507 rows | [194s]

Done: 48 shards | 9,275,507 training rows

Feature columns: 85
  First 10: ['age', 'age_of_diagnosis', 'basal', 'bmi', 'bolus', 'bolus_30m', 'carbs', 'carbs_1h', 'carbs_2h', 'cgm_15m_max']

Computing per-patient statistics...
  1183 patients with statistics
  Total features with patient stats: 89


0

In [ ]:
# ===========================================================================
# CELL 5: LightGBM Training — memory-efficient shard loading
# ===========================================================================
# Memory-efficient: load shards via polars lazy scan, collect one at a time
print('Loading training matrix (memory-efficient)...')

# Use polars lazy scan across all shards at once (reads column-by-column)
shard_glob = str(SHARD_DIR / '*.parquet')

# Determine which feature cols actually exist in shards
shard_schema = set(pd.read_parquet(shard_paths[0], engine='pyarrow').columns)
feat_in_shards = [c for c in FEATURE_COLS if c in shard_schema and c not in PAT_FEAT_COLS]
tgt_cols       = [f'target_{h}' for h in HORIZONS if f'target_{h}' in shard_schema]
load_cols      = ['id'] + feat_in_shards + tgt_cols

# Read with polars (much lower peak RAM than pandas concat)
print(f'  Scanning {len(shard_paths)} shards | {len(feat_in_shards)} features + {len(tgt_cols)} targets...')
lazy = pl.scan_parquet([str(sp) for sp in shard_paths]).select(load_cols)
df_all = lazy.collect().to_pandas()
del lazy
gc.collect()
print(f'  Loaded: {len(df_all):,} rows | {df_all.memory_usage().sum()/(1024**2):.0f} MB')

# Add patient statistics as personalisation features
df_all = df_all.merge(pat_stats, on='id', how='left')
for c in PAT_FEAT_COLS:
    df_all[c] = df_all[c].fillna(df_all[c].median()).astype(np.float32)

# Final feature list (only cols that actually exist)
ALL_FEATURE_COLS = [c for c in feat_in_shards + PAT_FEAT_COLS if c in df_all.columns]
# Add any missing expected features as zeros
for c in FEATURE_COLS + PAT_FEAT_COLS:
    if c not in df_all.columns:
        df_all[c] = np.float32(0.0)
ALL_FEATURE_COLS = FEATURE_COLS + PAT_FEAT_COLS

mem_mb = df_all.memory_usage(deep=False).sum() / (1024 ** 2)
print(f'  Matrix: {len(df_all):,} rows x {len(ALL_FEATURE_COLS)} features | {mem_mb:.0f} MB')

X       = df_all[ALL_FEATURE_COLS].fillna(0.0).to_numpy(np.float32)
targets = {h: df_all[f'target_{h}'].to_numpy(np.float32) for h in HORIZONS if f'target_{h}' in df_all.columns}
pat_ids = df_all['id'].values
del df_all
gc.collect()
print(f'  X shape: {X.shape} | RAM freed after del df_all')

# Patient-level train/val split (last 10% of patients = validation)
unique_pts = np.unique(pat_ids)
n_val      = max(1, int(len(unique_pts) * 0.10))
val_set    = set(unique_pts[-n_val:])
is_val     = np.array([p in val_set for p in pat_ids])
is_tr      = ~is_val
print(f'  Train: {int(is_tr.sum()):,} | Val: {int(is_val.sum()):,}')

lgbm_models = {}
print('\nTraining per-horizon models...')
for h in HORIZONS:
    if h not in targets:
        print(f'  h={h}min: target column missing — skipping')
        continue
    t_h = time.time()
    y   = targets[h]
    tr_mask = is_tr  & ~np.isnan(y) & (y > 0)
    vl_mask = is_val & ~np.isnan(y) & (y > 0)

    ds_tr = lgb.Dataset(X[tr_mask], label=y[tr_mask],
                        feature_name=ALL_FEATURE_COLS, free_raw_data=False)
    ds_vl = lgb.Dataset(X[vl_mask], label=y[vl_mask],
                        reference=ds_tr, free_raw_data=False)

    model = lgb.train(
        LGBM_PARAMS, ds_tr, num_boost_round=1000,
        valid_sets=[ds_vl],
        callbacks=[
            lgb.early_stopping(stopping_rounds=50, verbose=False),
            lgb.log_evaluation(period=200),
        ],
    )
    model.save_model(str(OUTPUT_DIR / f'lgbm_h{h}.lgb'))
    lgbm_models[h] = model

    yp = model.predict(X[vl_mask]).astype(np.float32)
    print(f'  h={h:3d}min | {model.num_trees():4d} trees | '
          f'MARD={mard(yp, y[vl_mask]):.2f}% | RMSE={rmse(yp, y[vl_mask]):.2f} | '
          f'[{time.time()-t_h:.0f}s]')
    del ds_tr, ds_vl, yp
    gc.collect()

del X, targets
gc.collect()
shutil.rmtree(SHARD_DIR)
print('\nAll models trained and saved.')


Loading training matrix (memory-efficient)...
  Scanning 48 shards | 85 features + 4 targets...
  Loaded: 9,275,507 rows | 3061 MB
  Matrix: 9,275,507 rows x 89 features | 3202 MB
  X shape: (9275507, 89) | RAM freed after del df_all
  Train: 9,027,987 | Val: 247,520

Training per-horizon models...
[200]	valid_0's l1: 16.9708
[400]	valid_0's l1: 16.827
[600]	valid_0's l1: 16.7692
[800]	valid_0's l1: 16.7387
[1000]	valid_0's l1: 16.7206
  h= 30min | 1000 trees | MARD=11.41% | RMSE=23.78 | [1496s]
[200]	valid_0's l1: 28.2593
[400]	valid_0's l1: 28.0096


In [ ]:
# ===========================================================================
# CELL 6: Predict on Test Set → submission_live_mard.parquet
#
# Critical correctness guarantees:
#  1. Only process patients present in the live template
#  2. Deduplicate test.parquet on (id, date) — 170K duplicate rows exist
#  3. submission = live_template.copy() — keys, dtypes, order from template
#  4. Three-way assertion (id, date, source_file) before saving
# ===========================================================================
print('Building test features...')
t0 = time.time()

# Normalise template IDs to strings for comparison
template_id_strs = set(live_template['id'].astype(str).unique())

# Load only patients that are in the template (268 of 309)
all_test_ids = pl.scan_parquet(TEST_PARQUET).select(
    pl.col('id').cast(pl.String)
).unique().collect()['id'].to_list()
test_ids = [p for p in all_test_ids if p in template_id_strs]
if not test_ids:
    print('WARNING: ID type mismatch — using all test patients as fallback')
    test_ids = all_test_ids
t_chunks = [test_ids[i:i + PATIENT_BATCH] for i in range(0, len(test_ids), PATIENT_BATCH)]
print(f'  Template patients in test: {len(test_ids)} | {len(t_chunks)} batches')

TEST_SHARD_DIR = OUTPUT_DIR / 'test_shards'
if TEST_SHARD_DIR.exists():
    shutil.rmtree(TEST_SHARD_DIR)
TEST_SHARD_DIR.mkdir(parents=True, exist_ok=True)

test_shard_paths = []
for idx, batch in enumerate(t_chunks):
    lazy = pl.scan_parquet(TEST_PARQUET).filter(
        pl.col('id').cast(pl.String).is_in(batch)
    )
    pdf  = _featurise_lazy(lazy, is_train=False, stride=1)
    pdf  = _add_iob_cob(pdf)
    sp   = TEST_SHARD_DIR / f'test_{idx:04d}.parquet'
    pdf.to_parquet(sp, index=False)
    test_shard_paths.append(sp)
    del pdf, lazy
    gc.collect()
    if (idx + 1) % max(1, len(t_chunks) // 5) == 0 or idx + 1 == len(t_chunks):
        print(f'  Batch {idx+1:3d}/{len(t_chunks)} [{time.time()-t0:.0f}s]')

df_test = pd.concat([pd.read_parquet(sp) for sp in test_shard_paths], ignore_index=True)
shutil.rmtree(TEST_SHARD_DIR)
gc.collect()
print(f'\nTest features loaded: {len(df_test):,} rows')

# Add patient statistics
df_test['id'] = df_test['id'].astype(str)   # normalise type
pat_stats_str = pat_stats.copy()
pat_stats_str['id'] = pat_stats_str['id'].astype(str)
df_test = df_test.merge(pat_stats_str, on='id', how='left')
for c in PAT_FEAT_COLS:
    fill_val = df_test[c].median() if df_test[c].notna().any() else 120.0
    df_test[c] = df_test[c].fillna(fill_val).astype(np.float32)

# Ensure all expected feature columns exist
for c in ALL_FEATURE_COLS:
    if c not in df_test.columns:
        df_test[c] = np.float32(0.0)

# Build string key for (id, date) matching
df_test['_key']       = df_test['id'].astype(str) + '||' + df_test['date'].astype(str)
live_template['_key'] = live_template['id'].astype(str) + '||' + live_template['date'].astype(str)

# Deduplicate test.parquet (contains ~170K duplicate timestamps)
n_before = len(df_test)
df_test  = df_test.drop_duplicates(subset='_key', keep='last').reset_index(drop=True)
print(f'Dedup: {n_before:,} → {len(df_test):,} (removed {n_before-len(df_test):,})')

# Check key overlap before merge
tmpl_keys = set(live_template['_key'])
test_keys = set(df_test['_key'])
overlap   = len(tmpl_keys & test_keys)
print(f'(id,date) overlap: {overlap:,} / {EXPECTED_ROWS:,}')
if overlap < EXPECTED_ROWS:
    print(f'WARNING: {EXPECTED_ROWS-overlap:,} template rows have no matching test row → will fill with CGM anchor')

# Align features to template row order via left join
merge_feats = ['_key', 'CGM_clean'] + [c for c in ALL_FEATURE_COLS if c in df_test.columns]
matched = live_template[['id', 'source_file', 'date', '_key']].merge(
    df_test[list(set(merge_feats))].drop_duplicates('_key'),
    on='_key', how='left'
).drop(columns=['_key'])

assert len(matched) == EXPECTED_ROWS, \
    f'Merge row count {len(matched):,} != expected {EXPECTED_ROWS:,}'

# Fill NaN features with 0.0
for c in ALL_FEATURE_COLS:
    matched[c] = matched[c].fillna(0.0).astype(np.float32)

# CGM anchor for physiological clipping
cgm_anchor = matched['CGM_clean'].fillna(120.0).values.astype(np.float64)
X_test     = matched[ALL_FEATURE_COLS].to_numpy(np.float32)
print(f'Prediction matrix: {X_test.shape}')

# ── WRITE PREDICTIONS INTO TEMPLATE ──────────────────────────────────────
# Start from live_template.copy() — id/source_file/date/dtypes preserved exactly
# This is the only approach that passes leaderboard key validation.
submission = live_template.copy().reset_index(drop=True)
submission.drop(columns=['_key'], inplace=True, errors='ignore')

for h in HORIZONS:
    raw = lgbm_models[h].predict(X_test).astype(np.float64)
    # Replace NaN predictions with CGM anchor
    raw = np.where(np.isnan(raw), cgm_anchor, raw)
    # Physiological clipping: glucose cannot change faster than MAX_ROC_PER_MIN
    max_delta = MAX_ROC_PER_MIN * h
    lo  = np.clip(cgm_anchor - max_delta, GLUCOSE_MIN, GLUCOSE_MAX)
    hi  = np.clip(cgm_anchor + max_delta, GLUCOSE_MIN, GLUCOSE_MAX)
    submission[f'pred_{h}'] = np.clip(raw, lo, hi)

# Ensure correct column order
submission = submission[['id', 'source_file', 'date', 'pred_30', 'pred_60', 'pred_90', 'pred_120']]

# ── FULL VALIDATION (mirrors official run.py checks) ─────────────────────
tmpl_clean = live_template.drop(columns=['_key'], errors='ignore')
null_count = submission[['pred_30','pred_60','pred_90','pred_120']].isnull().sum().sum()
assert len(submission) == EXPECTED_ROWS,                                        'Row count wrong!'
assert null_count == 0,                                                          f'{null_count} NaN!'
assert (submission['id'].values == tmpl_clean['id'].values).all(),               'ID mismatch!'
assert (submission['date'].values == tmpl_clean['date'].values).all(),           'Date mismatch!'
assert (submission['source_file'].values == tmpl_clean['source_file'].values).all(), 'SF mismatch!'

# Save
OUT = OUTPUT_DIR / 'submission_live_mard.parquet'
submission.to_parquet(OUT, index=False)

print(f'\n✓ SAVED: {OUT.name}')
print(f'  Rows   : {len(submission):,}  (expected {EXPECTED_ROWS:,})')
print(f'  NaN    : {null_count}')
print(f'  Keys   : id ✓ | date ✓ | source_file ✓')
for col in ['pred_30','pred_60','pred_90','pred_120']:
    print(f'  {col}: [{submission[col].min():.1f}, {submission[col].max():.1f}]')

del df_test, matched, X_test
gc.collect()
display(submission.head(5))

In [ ]:
# ===========================================================================
# CELL 7: Local Scoring (targets.parquet is available in toolkit dataset)
# ===========================================================================
if LIVE_TARGETS_PATH.exists():
    print('Scoring locally against targets.parquet...')
    tgts = pd.read_parquet(LIVE_TARGETS_PATH)
    tgts['date'] = pd.to_datetime(tgts['date'])

    scored = submission.merge(
        tgts[['id','date','target_30','target_60','target_90','target_120']],
        on=['id','date'], how='left'
    )

    print('\n' + '='*55)
    print('  LOCAL EVALUATION (targets.parquet)')
    print('='*55)
    for h in HORIZONS:
        p = scored[f'pred_{h}'].values
        t = scored[f'target_{h}'].values
        print(f'  {h:3d}-min | MARD={mard(p,t):6.2f}% | RMSE={rmse(p,t):6.2f} mg/dL')
    print('='*55)
    print('  Rank 1 target  | MARD=  10.70% | RMSE=  23.00 mg/dL')
    print('='*55)
else:
    print('targets.parquet not found — scores will appear on leaderboard after submission.')

# List all output files
print('\nOutput files:')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        print(f'  {f.name:<50s}  {f.stat().st_size / (1024**2):.1f} MB')

print('\nDownload submission_live_mard.parquet and submit at:')
print('  https://metabonetglucose-leaderboard.hf.space/?tab=live-leaderboard-tab&view=submit')